# Chocolate Sales Performance Analysis

**Business question:** Which reps, products, and regions are driving revenue — and where are the opportunities?

Every Monday morning, a sales manager needs answers to three questions:
1. Is revenue trending up or down?
2. Who are my top (and bottom) performers?
3. Which products and markets should I double down on?

This notebook uses transactional chocolate sales data to answer exactly those questions.

In [4]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [5]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.load import load_data
from src.metrics import monthly_revenue, mom_growth, rep_summary, country_summary, product_summary

px.defaults.template = 'plotly_white'

## 1. Data Overview

In [6]:
df = load_data()

print(f"Rows: {len(df):,}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Sales reps: {df['Sales Person'].nunique()}")
print(f"Countries: {df['Country'].nunique()}")
print(f"Products: {df['Product'].nunique()}")
print(f"\nNull values:\n{df.isnull().sum()}")

Rows: 1,094
Date range: 2022-01-03 → 2022-08-31
Sales reps: 25
Countries: 6
Products: 22

Null values:
Sales Person       0
Country            0
Product            0
Date               0
Amount             0
Boxes Shipped      0
revenue_per_box    0
dtype: int64


In [7]:
df.head()

,Sales Person,Country,Product,Date,Amount,Boxes Shipped,revenue_per_box
0,Jehu Rudeforth,UK,Mint Chip Choco,2022-01-04,5320.0,180,29.555556
1,Van Tuxwell,India,85% Dark Bars,2022-08-01,7896.0,94,84.000000
2,Gigi Bohling,India,Peanut Butter Cubes,2022-07-07,4501.0,91,49.461538
3,Jan Morforth,Australia,Peanut Butter Cubes,2022-04-27,12726.0,342,37.210526
4,Jehu Rudeforth,UK,Peanut Butter Cubes,2022-02-24,13685.0,184,74.375000


## 2. Revenue Trends

Monthly revenue with month-over-month growth rate.

In [8]:
monthly = mom_growth(monthly_revenue(df))

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Bar(x=monthly['Month'], y=monthly['Revenue'], name='Revenue', marker_color='steelblue', opacity=0.7),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=monthly['Month'], y=monthly['MoM Growth %'], name='MoM Growth %',
               mode='lines+markers', line=dict(color='orange', width=2)),
    secondary_y=True
)

fig.update_layout(title='Monthly Revenue & MoM Growth', hovermode='x unified')
fig.update_yaxes(title_text='Revenue ($)', secondary_y=False)
fig.update_yaxes(title_text='MoM Growth (%)', secondary_y=True)
fig.show()

In [9]:
# Summary stats
print(f"Total revenue: ${df['Amount'].sum():,.0f}")
print(f"Avg monthly revenue: ${monthly['Revenue'].mean():,.0f}")
print(f"Best month: {monthly.loc[monthly['Revenue'].idxmax(), 'Month'].strftime('%b %Y')} (${monthly['Revenue'].max():,.0f})")
print(f"Worst month: {monthly.loc[monthly['Revenue'].idxmin(), 'Month'].strftime('%b %Y')} (${monthly['Revenue'].min():,.0f})")

Total revenue: $6,183,625
Avg monthly revenue: $772,953
Best month: Jan 2022 ($896,105)
Worst month: Apr 2022 ($674,051)


## 3. Sales Rep Performance

Total revenue by rep, colored by revenue per box (deal quality).

In [10]:
reps_df = rep_summary(df)

fig = px.bar(
    reps_df,
    x='Revenue', y='Sales Person',
    orientation='h',
    color='Revenue per Box',
    color_continuous_scale='RdYlGn',
    title='Sales Rep Leaderboard (color = revenue per box)',
    labels={'Revenue': 'Total Revenue ($)', 'Sales Person': '', 'Revenue per Box': '$/Box'},
    text=reps_df['Revenue'].apply(lambda x: f'${x:,.0f}')
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.update_traces(textposition='outside')
fig.show()

In [11]:
# Consistency: how volatile is each rep month-to-month?
rep_monthly = (
    df.set_index('Date')
    .groupby('Sales Person')
    .resample('ME')['Amount']
    .sum()
    .reset_index()
)
rep_consistency = (
    rep_monthly.groupby('Sales Person')['Amount']
    .agg(Mean='mean', Std='std')
    .assign(CV=lambda x: x['Std'] / x['Mean'])  # coefficient of variation = lower is more consistent
    .sort_values('CV')
    .reset_index()
)
print("Most consistent reps (low CV = reliable month-to-month):")
print(rep_consistency.head(5).to_string(index=False))

Most consistent reps (low CV = reliable month-to-month):
       Sales Person      Mean          Std       CV
       Barr Faughny 32339.125  6681.284766 0.206601
       Ches Bonnell 40112.625 14557.767587 0.362922
Dennison Crosswaite 36458.625 13314.149283 0.365185
       Husein Augar 25651.500  9583.480266 0.373603
      Kelci Walkden 38963.750 14814.625358 0.380216


## 4. Geographic Breakdown

In [12]:
country_df = country_summary(df)

fig = px.bar(
    country_df,
    x='Country', y='Revenue',
    color='Revenue',
    color_continuous_scale='Blues',
    title='Revenue by Country',
    text=country_df['Revenue'].apply(lambda x: f'${x:,.0f}'),
    labels={'Revenue': 'Total Revenue ($)'}
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 5. Product Performance

In [13]:
product_df = product_summary(df)

fig = px.treemap(
    product_df,
    path=['Product'],
    values='Revenue',
    color='Revenue per Box',
    color_continuous_scale='RdYlGn',
    title='Product Revenue (size) vs. Deal Quality (color = $/box)',
    hover_data={'Revenue': ':,.0f', 'Revenue per Box': ':.2f'}
)
fig.show()

In [14]:
print("Top 5 products by revenue:")
print(product_df.head(5)[['Product', 'Revenue', 'Boxes', 'Revenue per Box']].to_string(index=False))

Top 5 products by revenue:
            Product  Revenue  Boxes  Revenue per Box
 Smooth Sliky Salty 349692.0   8810        39.692622
     50% Dark Bites 341712.0   9792        34.897059
         White Choc 329147.0   8240        39.945024
Peanut Butter Cubes 324842.0   8304        39.118738
            Eclairs 312445.0   8757        35.679456


## 6. Deal Size Trends

Revenue per box over time — are we shipping larger or smaller orders?

In [15]:
deal_size = (
    df.set_index('Date')
    .resample('ME')[['Amount', 'Boxes Shipped']]
    .sum()
    .assign(revenue_per_box=lambda x: x['Amount'] / x['Boxes Shipped'])
    .reset_index()
)

fig = px.line(
    deal_size,
    x='Date', y='revenue_per_box',
    markers=True,
    title='Avg Revenue per Box (Monthly)',
    labels={'revenue_per_box': 'Revenue per Box ($)', 'Date': ''}
)
fig.show()

## 7. Key Takeaways

- **Revenue:** Jan peak → Apr trough → Jun recovery pattern                                                                                                                  
- **Reps:** Rafaelita Blaksland flagged as hidden high-value seller; Karlen McCaffrey flagged as high-volume/low-quality worth coaching
- **Geography:** Balanced across 6 markets; USA has highest avg deal size                                                                                                    
- **Products:** Tight competition at the top; Peanut Butter Cubes is the most balanced on both volume and margin